In [ ]:
import geopandas as gpd


#Load data — all analysis in EPSG:2169 (metres)
quarters   = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Lux_quaters.geojson").to_crs(epsg=2169)
commercial = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\Commerical_land.gpkg").to_crs(epsg=2169)
stops      = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\stop_freq_avl.gpkg").to_crs(epsg=2169)

#Intersect commercial area with quarter area and compute the result per quarter
commercial_intersect = gpd.overlay(quarters, commercial, how="intersection")
commercial_intersect["comm_area"] = commercial_intersect.area

#Create a DataFrame summing the commercial area per quarter
comm_area_quarter = commercial_intersect.groupby("FK_QUART_NAME")["comm_area"].sum().reset_index()

#Find the total area of each quarter
quarters["total_area"] = quarters.geometry.area

#Calculate coverage ratio: comm_area / total_area
quarters = quarters.merge(comm_area_quarter, on="FK_QUART_NAME", how="left").fillna(0)
quarters["coverage"] = quarters["comm_area"] / quarters["total_area"]

#Select the top five coverage ratios
top5 = quarters.sort_values("coverage", ascending=False).head(5)

top_comm = gpd.overlay(commercial, top5, how="intersection")

#Filter out polygons that are too small
top_comm = top_comm[(top_comm.geometry.area > 1000)]

#Find candidate stops for each quarter — all stops within 100m of each commercial polygon
top_comm_buffered = top_comm.copy()
top_comm_buffered["geometry"] = top_comm.geometry.buffer(100)

stops_near = gpd.sjoin(stops, top_comm_buffered, how="inner", predicate="within")
stops_near = stops_near[["FK_QUART_NAME", "stop_id", "geometry"]].drop_duplicates(subset=["FK_QUART_NAME", "stop_id"])
stops_near["node"] = "pt_" + stops_near["stop_id"].astype(str)

#Reproject map outputs to EPSG:4326 for QGIS
top_districts = top5[["FK_QUART_NAME", "coverage", "geometry"]].to_crs(epsg=4326)
top_comm      = top_comm.to_crs(epsg=4326)

#========================
#Save outputs
#========================
top_districts.to_file("business_districts.gpkg", driver="GPKG")
stops_near.to_csv("district_candidate_stops.csv", index=False)
top_comm.to_file("top_district_commercial.gpkg", driver="GPKG")

print("Business district stops and quarters formed")